In [1]:
import os
import json
import math
from pathlib import Path

import pandas as pd
import numpy as np
from astropy.table import Table, Column, MaskedColumn
from astropy.io import fits
from astropy.visualization import ZScaleInterval
from astropy.coordinates import SkyCoord
from astropy import units as u
import matplotlib.pyplot as plt
from ztf_downloads.ztf_search import metadata_search_resumable
from ztf_downloads.ztf_download import build_sci_url,build_cutout_url, batch_download_resumable,_safe_label

# Load galaxy catalog

final_class_table_new_(only_in_dist_bounds).csv - galaxy list table

In [2]:
id_cols = {
    'objID_SDSS-DR16': 'string',
    'id_DESI-DR8': 'string',
    'objid_DESI-DR8': 'string',
    'objname_NED-LVS': 'string',
    'Source_GAIA-DR3': 'string',
    'Name_CatWISE': 'string',
    'WISEA_CatWISE': 'string',
}

galaxies = pd.read_csv('ZTF22aabjpxh_galaxy_catalog.csv', 
                       dtype=id_cols, low_memory=False)

# Add a 1‑based row number as the folder name
galaxies['galaxy_row'] = galaxies.index + 1   # becomes 1,2,3,...

# Coordinates of all galaxies
gal_coords = SkyCoord(
    ra=galaxies["ra_fin"].to_numpy() * u.deg,
    dec=galaxies["dec_fin"].to_numpy() * u.deg,
)

# After loading galaxies, define coords
coords = [(r, d) for r, d in zip(galaxies['ra_fin'], galaxies['dec_fin'])]

C:\Users\masha\AppData\Local\Temp\ipykernel_34744\209835884.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  galaxies['galaxy_row'] = galaxies.index + 1   # becomes 1,2,3,...


In [ ]:
file_path = 'ZTF22aabjpxh_galaxy_catalog.csv'
galaxies = pd.read_csv(file_path, low_memory=False)

# Add galaxy_row (1‑based index)
galaxies.insert(0, 'object_id', range(1, len(galaxies) + 1))

# Save back
galaxies.to_csv('numbered_galaxies.csv', index=False)
print(f"Added object_id as first column to numbered_galaxies.csv")

C:\Users\masha\AppData\Local\Temp\ipykernel_34744\3382199402.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  galaxies.insert(0, 'galaxy_row', range(1, len(galaxies) + 1))


Added galaxy_row as first column to numbered_galaxies.csv


In [4]:
print(len(galaxies), 'galaxies in localization volume')

46365 galaxies in localization volume


# Download difference image metadata

In [12]:
diff_meta_file = "metadata_ZTF22aabjpxh.csv"
diff_meta_progress = "metadata_ZTF22aabjpxh.progress.json"

df_meta = metadata_search_resumable(
    positions=coords,
    output_csv=diff_meta_file,
    progress_json=diff_meta_progress,
    batch_size=100,
    size_deg=0.01,
    product_type="sci",
    filtercodes=["zr", "zi"],
    date_start="2022-02-19", # TO-DO: automatize with config
    date_end="2022-02-21", # TO-DO: automatize with +2 days
    timeout=300,
 )

print(f"Current metadata rows in {diff_meta_file}: {len(df_meta)}")

Metadata batch 347/464 (34600:34700) attempt 1
Metadata batch 348/464 (34700:34800) attempt 1
Metadata batch 349/464 (34800:34900) attempt 1
Metadata batch 350/464 (34900:35000) attempt 1
Retrying in 5s due to: 502 Server Error: Bad Gateway for url: https://irsa.ipac.caltech.edu/ibe/search/ztf/products/sci
Metadata batch 350/464 (34900:35000) attempt 2
Metadata batch 351/464 (35000:35100) attempt 1
Metadata batch 352/464 (35100:35200) attempt 1
Metadata batch 353/464 (35200:35300) attempt 1
Metadata batch 354/464 (35300:35400) attempt 1
Metadata batch 355/464 (35400:35500) attempt 1
Metadata batch 356/464 (35500:35600) attempt 1
Metadata batch 357/464 (35600:35700) attempt 1
Metadata batch 358/464 (35700:35800) attempt 1
Metadata batch 359/464 (35800:35900) attempt 1
Metadata batch 360/464 (35900:36000) attempt 1
Metadata batch 361/464 (36000:36100) attempt 1
Metadata batch 362/464 (36100:36200) attempt 1
Metadata batch 363/464 (36200:36300) attempt 1
Metadata batch 364/464 (36300:3640

In [13]:
# Keep a convenience copy of the current resumable metadata file
df_meta.to_csv("metadata_ZTF22aabjpxh.csv", index=False)
print(f"Saved snapshot rows: {len(df_meta)}")

Saved snapshot rows: 136585


In [6]:
# Load the resumable metadata file by default
df_meta = pd.read_csv("metadata_ZTF22aabjpxh.csv")
print(f"Loaded rows: {len(df_meta)}")

Loaded rows: 136585


In [ ]:
# ref_meta_file = "metadata_ref_ZTF22aabjpxh.csv"
# ref_meta_progress = "metadata_ref_ZTF22aabjpxh.json"

# ref_meta = metadata_search_resumable(
#     positions=coords,
#     output_csv=ref_meta_file,
#     progress_json=ref_meta_progress,
#     batch_size=100,
#     size_deg=0.01,
#     product_type="ref",
#     filtercodes=["zr", "zi"],
#     timeout=300,
#  )

# print(f"Current metadata rows in {ref_meta_file}: {len(ref_meta)}")

Metadata batch 139/269 (13800:13900) attempt 1
Retrying in 5s due to: HTTPSConnectionPool(host='irsa.ipac.caltech.edu', port=443): Read timed out.
Metadata batch 139/269 (13800:13900) attempt 2
Metadata batch 140/269 (13900:14000) attempt 1
Metadata batch 141/269 (14000:14100) attempt 1
Metadata batch 142/269 (14100:14200) attempt 1
Metadata batch 143/269 (14200:14300) attempt 1
Metadata batch 144/269 (14300:14400) attempt 1
Metadata batch 145/269 (14400:14500) attempt 1
Metadata batch 146/269 (14500:14600) attempt 1
Metadata batch 147/269 (14600:14700) attempt 1
Metadata batch 148/269 (14700:14800) attempt 1
Metadata batch 149/269 (14800:14900) attempt 1
Metadata batch 150/269 (14900:15000) attempt 1
Metadata batch 151/269 (15000:15100) attempt 1
Metadata batch 152/269 (15100:15200) attempt 1
Metadata batch 153/269 (15200:15300) attempt 1
Metadata batch 154/269 (15300:15400) attempt 1
Metadata batch 155/269 (15400:15500) attempt 1
Metadata batch 156/269 (15500:15600) attempt 1
Metadat

In [ ]:
# # Keep a convenience copy of the current resumable metadata file
# ref_meta.to_csv("metadata_ref_v0_4.csv", index=False)
# print(f"Saved snapshot rows: {len(df_meta)}")

In [ ]:
# # Load the resumable metadata file by default
# ref_meta = pd.read_csv("metadata_ref_v0_4.csv")
# print(f"Loaded rows: {len(df_meta)}")

# Download difference images in filters r, i 

warning: this may take around 16 hours and produce 40 Gb of data

In [7]:
# -------------------------------
# Load and filter ZTF metadata
# -------------------------------
allowed_filters = {"zr", "zi"}
df_meta = df_meta[df_meta["filtercode"].isin(allowed_filters)].copy()
print(f"Rows after filter restriction (zr, zi): {len(df_meta)}")

# Coordinates of ZTF difference images
meta_coords = SkyCoord(
    ra=df_meta["in_ra"].to_numpy() * u.deg,
    dec=df_meta["in_dec"].to_numpy() * u.deg,
)

# Match each metadata row to the nearest galaxy
match_idx, sep2d, _ = meta_coords.match_to_catalog_sky(gal_coords)
tol = 2.0 * u.arcsec
is_match = sep2d <= tol

# Assign the galaxy's row number (folder name) to matched rows
matched_folders = galaxies.iloc[match_idx]["galaxy_row"].astype(str).to_numpy()
df_meta["gal_folder"] = pd.Series(
    np.where(is_match, matched_folders, "unmatched"), 
    index=df_meta.index
)

print(f"df_meta length: {len(df_meta)}")
print(f"Rows with galaxy match (<= {tol}): {is_match.sum()}")
print(f"Unmatched rows: {(~is_match).sum()}")
print(f"Max separation: {sep2d.max().to(u.arcsec):.3f}")

# (Optional) Also store the original objID for reference
df_meta["objID_SDSS-DR16"] = pd.Series(
    np.where(is_match, galaxies.iloc[match_idx]["objID_SDSS-DR16"].astype(str), pd.NA),
    index=df_meta.index
)

# -------------------------------
# Prepare download tasks
# -------------------------------
# Helper to sanitize folder names (though numbers are safe)
def _safe_label(label):
    return str(label).replace('/', '_').replace('\\', '_')

df_meta['gal_folder'] = df_meta['gal_folder'].apply(_safe_label)

# Create unique file names within each folder (pipeline-compatible with full suffix)
df_meta['file_label'] = df_meta.apply(
    lambda row: (
        f"{row['gal_folder']}_RA{row['in_ra']:.6f}_DEC{row['in_dec']:.6f}_{row['filtercode']}_{str(int(row['filefracday'])).zfill(14)}__ztf_{int(row['field']):06d}_{row['filtercode']}_c{int(row['ccdid'])}_o_q{int(row['qid'])}_scimrefdiffimg.fits.fz"
    ),
    axis=1
)

from urllib.parse import urlparse
from pathlib import Path

cutout_inputs = []
for _, row in df_meta.iterrows():
    # 1. Get the correct science image URL using the existing function
    base_sci_url = build_sci_url(row, suffix="scimrefdiffimg.fits.fz")
    
    # 2. Extract the original filename from that URL
    original_fname = Path(urlparse(base_sci_url).path).name
    
    # 3. Create cutout URL (adds center, size, gzip)
    url = build_cutout_url(base_sci_url, row["in_ra"], row["in_dec"], size_arcsec=240)
    
    # 4. Get the full 14‑digit filefracday as string (ensure no truncation)
    filefracday_str = str(int(row["filefracday"])).zfill(14)
    
    # 5. Your custom label
    custom_label = f"RA{row['in_ra']:.4f}_DEC{row['in_dec']:.4f}_{row['filtercode']}_{filefracday_str}"
    
    # 6. Desired final filename: gal_folder + custom_label + "__" + original_fname
    desired_fname = f"{row['gal_folder']}_{custom_label}__{original_fname}"
    
    cutout_inputs.append((url, desired_fname, row['gal_folder']))
# -------------------------------
# Batch download (resumable)
# -------------------------------
batch_download_resumable(
    cutout_inputs,
    out_dir="diff_ZTF22aabjpxh",   # parent directory
    progress_json="diff_ZTF22aabjpxh.progress.json",
    max_workers=9,
    chunk_size=200,
)

# List all downloaded FITS files (now nested in subfolders)
diff_cutouts = sorted(str(p) for p in Path("diff_ZTF22aabjpxh").rglob("*.fits*"))
print(f"Diff files currently on disk: {len(diff_cutouts)}")

Rows after filter restriction (zr, zi): 103470
df_meta length: 103470
Rows with galaxy match (<= 2.0 arcsec): 103470
Unmatched rows: 0
Max separation: 0.000 arcsec
All chunks complete: 103470 items
Diff files currently on disk: 103470
